In [ ]:
# ============================================================
# DAY 15 - EXECUTIVE HOTEL BOOKING EDA
# Complete End-to-End Exploratory Data Analysis
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "/mnt/data/Day15_Executive_Hotel_Booking_EDA_Dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

# ============================================================
# 3. INITIAL DATA UNDERSTANDING
# ============================================================

print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== LAST 5 ROWS ==========")
display(df.tail())

print("\n========== DATASET SHAPE ==========")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFO ==========")
df.info()

print("\n========== DESCRIPTIVE STATISTICS ==========")
display(df.describe(include="all").T)

# ============================================================
# 4. DATA QUALITY ASSESSMENT
# ============================================================

# Missing values
print("\n========== MISSING VALUES ==========")

missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

missing = missing[missing["Missing_Count"] > 0]
missing = missing.sort_values("Missing_Count", ascending=False)

display(missing)

# Duplicate records
print("\n========== DUPLICATE RECORDS ==========")

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

print("Duplicate Booking IDs:", df["Booking_ID"].duplicated().sum())

# ============================================================
# 5. CHECK UNIQUE / INCONSISTENT CATEGORICAL VALUES
# ============================================================

categorical_columns = df.select_dtypes(include="object").columns

print("\n========== UNIQUE CATEGORICAL VALUES ==========")

for col in categorical_columns:
    print("\n", col)
    print(df[col].dropna().unique())

# ============================================================
# 6. DATA CLEANING
# ============================================================

df_clean = df.copy()

# ------------------------------------------------------------
# Remove duplicate records
# ------------------------------------------------------------

df_clean = df_clean.drop_duplicates()

print("\nRows after removing duplicates:", len(df_clean))

# ------------------------------------------------------------
# Standardize string columns
# ------------------------------------------------------------

string_columns = df_clean.select_dtypes(include="object").columns

for col in string_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()

# ------------------------------------------------------------
# Standardize Hotel Type
# ------------------------------------------------------------

df_clean["Hotel_Type"] = (
    df_clean["Hotel_Type"]
    .str.strip()
    .str.title()
)

# ------------------------------------------------------------
# Standardize Market Segment
# ------------------------------------------------------------

df_clean["Market_Segment"] = (
    df_clean["Market_Segment"]
    .str.strip()
    .str.title()
)

# Convert Online Ta variations
df_clean["Market_Segment"] = df_clean["Market_Segment"].replace({
    "Online Ta": "Online TA",
    "Offline Ta/To": "Offline TA/TO"
})

# ------------------------------------------------------------
# Standardize Meal Type
# ------------------------------------------------------------

df_clean["Meal_Type"] = (
    df_clean["Meal_Type"]
    .str.strip()
    .str.upper()
)

df_clean["Meal_Type"] = df_clean["Meal_Type"].replace({
    "B&B": "BB",
    "UNKNOWN": np.nan,
    "UNDEFINED": np.nan
})

# ------------------------------------------------------------
# Standardize Reservation Status
# ------------------------------------------------------------

df_clean["Reservation_Status"] = (
    df_clean["Reservation_Status"]
    .str.strip()
    .str.title()
)

# ------------------------------------------------------------
# Standardize missing/unknown values
# ------------------------------------------------------------

df_clean = df_clean.replace(
    ["Unknown", "unknown", "UNKNOWN", ""],
    np.nan
)

# ------------------------------------------------------------
# Convert date columns
# ------------------------------------------------------------

date_columns = [
    "Booking_Date",
    "Arrival_Date",
    "Reservation_Status_Date"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Fill missing numerical values
# ------------------------------------------------------------

numerical_columns = df_clean.select_dtypes(
    include=np.number
).columns

for col in numerical_columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(
            df_clean[col].median()
        )

# ------------------------------------------------------------
# Fill missing categorical values
# ------------------------------------------------------------

categorical_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_columns:
    if df_clean[col].isnull().sum() > 0:
        mode_value = df_clean[col].mode()

        if len(mode_value) > 0:
            df_clean[col] = df_clean[col].fillna(
                mode_value[0]
            )

# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

# Total guests
df_clean["Total_Guests"] = (
    df_clean["Adults"]
    + df_clean["Children"]
    + df_clean["Babies"]
)

# Total stay nights
df_clean["Total_Nights"] = (
    df_clean["Weekend_Nights"]
    + df_clean["Weekday_Nights"]
)

# Total stay revenue
df_clean["Estimated_Revenue"] = (
    df_clean["ADR"] *
    df_clean["Total_Nights"]
)

# Arrival year
df_clean["Arrival_Year"] = (
    df_clean["Arrival_Date"].dt.year
)

# Arrival month
df_clean["Arrival_Month"] = (
    df_clean["Arrival_Date"].dt.month
)

# Arrival month name
df_clean["Arrival_Month_Name"] = (
    df_clean["Arrival_Date"].dt.month_name()
)

# Arrival day of week
df_clean["Arrival_Day"] = (
    df_clean["Arrival_Date"].dt.day_name()
)

# Cancellation label
df_clean["Cancellation_Label"] = np.where(
    df_clean["Is_Canceled"] == 1,
    "Canceled",
    "Not Canceled"
)

# ============================================================
# 8. OUTLIER DETECTION
# ============================================================

print("\n========== OUTLIER ANALYSIS ==========")

outlier_columns = [
    "Lead_Time_Days",
    "Weekend_Nights",
    "Weekday_Nights",
    "Adults",
    "Children",
    "ADR",
    "Total_Nights",
    "Estimated_Revenue"
]

outlier_summary = []

for col in outlier_columns:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = (
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ).sum()

    percentage = count / len(df_clean) * 100

    outlier_summary.append([
        col,
        Q1,
        Q3,
        lower,
        upper,
        count,
        percentage
    ])

outlier_df = pd.DataFrame(
    outlier_summary,
    columns=[
        "Column",
        "Q1",
        "Q3",
        "Lower_Bound",
        "Upper_Bound",
        "Outlier_Count",
        "Outlier_Percentage"
    ]
)

display(outlier_df)

# ============================================================
# 9. DESCRIPTIVE STATISTICS
# ============================================================

print("\n========== NUMERICAL SUMMARY ==========")

display(
    df_clean[
        outlier_columns
    ].describe().T
)

# ============================================================
# 10. UNIVARIATE ANALYSIS
# ============================================================

# ------------------------------------------------------------
# Hotel Type
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

sns.countplot(
    data=df_clean,
    x="Hotel_Type"
)

plt.title("Bookings by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Cancellation
# ------------------------------------------------------------

plt.figure(figsize=(7,5))

sns.countplot(
    data=df_clean,
    x="Cancellation_Label"
)

plt.title("Booking Cancellation Distribution")
plt.xlabel("Booking Status")
plt.ylabel("Number of Bookings")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Market Segment
# ------------------------------------------------------------

plt.figure(figsize=(10,5))

sns.countplot(
    data=df_clean,
    y="Market_Segment",
    order=df_clean["Market_Segment"].value_counts().index
)

plt.title("Bookings by Market Segment")
plt.xlabel("Number of Bookings")
plt.ylabel("Market Segment")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Customer Type
# ------------------------------------------------------------

plt.figure(figsize=(9,5))

sns.countplot(
    data=df_clean,
    x="Customer_Type",
    order=df_clean["Customer_Type"].value_counts().index
)

plt.title("Bookings by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# ADR Distribution
# ------------------------------------------------------------

plt.figure(figsize=(9,5))

sns.histplot(
    data=df_clean,
    x="ADR",
    bins=40,
    kde=True
)

plt.title("Distribution of Average Daily Rate (ADR)")
plt.xlabel("ADR")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Lead Time Distribution
# ------------------------------------------------------------

plt.figure(figsize=(9,5))

sns.histplot(
    data=df_clean,
    x="Lead_Time_Days",
    bins=40,
    kde=True
)

plt.title("Distribution of Lead Time")
plt.xlabel("Lead Time (Days)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Total Nights
# ------------------------------------------------------------

plt.figure(figsize=(9,5))

sns.histplot(
    data=df_clean,
    x="Total_Nights",
    bins=30,
    kde=True
)

plt.title("Distribution of Total Stay Nights")
plt.xlabel("Total Nights")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ============================================================
# 11. BOX PLOTS - OUTLIER VISUALIZATION
# ============================================================

plt.figure(figsize=(9,5))

sns.boxplot(
    data=df_clean,
    x="ADR"
)

plt.title("ADR Outlier Analysis")
plt.xlabel("Average Daily Rate")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))

sns.boxplot(
    data=df_clean,
    x="Lead_Time_Days"
)

plt.title("Lead Time Outlier Analysis")
plt.xlabel("Lead Time (Days)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))

sns.boxplot(
    data=df_clean,
    x="Total_Nights"
)

plt.title("Total Stay Nights Outlier Analysis")
plt.xlabel("Total Nights")
plt.tight_layout()
plt.show()

# ============================================================
# 12. BIVARIATE ANALYSIS
# ============================================================

# ------------------------------------------------------------
# Hotel Type vs Cancellation
# ------------------------------------------------------------

hotel_cancel = pd.crosstab(
    df_clean["Hotel_Type"],
    df_clean["Cancellation_Label"],
    normalize="index"
) * 100

display(hotel_cancel)

hotel_cancel.plot(
    kind="bar",
    figsize=(9,5)
)

plt.title("Cancellation Rate by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("Percentage")
plt.xticks(rotation=20)
plt.legend(title="Booking Status")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Market Segment vs Cancellation
# ------------------------------------------------------------

segment_cancel = pd.crosstab(
    df_clean["Market_Segment"],
    df_clean["Cancellation_Label"],
    normalize="index"
) * 100

display(segment_cancel)

segment_cancel.plot(
    kind="bar",
    figsize=(11,6)
)

plt.title("Cancellation Rate by Market Segment")
plt.xlabel("Market Segment")
plt.ylabel("Percentage")
plt.xticks(rotation=30)
plt.legend(title="Booking Status")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Lead Time vs Cancellation
# ------------------------------------------------------------

plt.figure(figsize=(9,6))

sns.boxplot(
    data=df_clean,
    x="Cancellation_Label",
    y="Lead_Time_Days"
)

plt.title("Lead Time vs Booking Cancellation")
plt.xlabel("Booking Status")
plt.ylabel("Lead Time (Days)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# ADR by Hotel Type
# ------------------------------------------------------------

plt.figure(figsize=(8,6))

sns.boxplot(
    data=df_clean,
    x="Hotel_Type",
    y="ADR"
)

plt.title("ADR Distribution by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("ADR")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Stay Duration vs ADR
# ------------------------------------------------------------

plt.figure(figsize=(9,6))

sns.scatterplot(
    data=df_clean,
    x="Total_Nights",
    y="ADR",
    alpha=0.5
)

plt.title("Relationship Between Stay Duration and ADR")
plt.xlabel("Total Nights")
plt.ylabel("ADR")
plt.tight_layout()
plt.show()

# ============================================================
# 13. GROUP-WISE ANALYSIS
# ============================================================

# Average ADR by hotel
print("\n========== AVERAGE ADR BY HOTEL ==========")

avg_adr_hotel = (
    df_clean
    .groupby("Hotel_Type")["ADR"]
    .agg(["mean", "median", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(avg_adr_hotel)

# Average revenue by hotel
print("\n========== REVENUE BY HOTEL ==========")

revenue_hotel = (
    df_clean
    .groupby("Hotel_Type")["Estimated_Revenue"]
    .agg(["sum", "mean", "median"])
    .sort_values("sum", ascending=False)
)

display(revenue_hotel)

# Cancellation rate by hotel
print("\n========== CANCELLATION RATE BY HOTEL ==========")

cancellation_hotel = (
    df_clean
    .groupby("Hotel_Type")["Is_Canceled"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(cancellation_hotel)

# Cancellation rate by market segment
print("\n========== CANCELLATION RATE BY MARKET SEGMENT ==========")

cancellation_segment = (
    df_clean
    .groupby("Market_Segment")["Is_Canceled"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(cancellation_segment)

# Average revenue by customer type
print("\n========== REVENUE BY CUSTOMER TYPE ==========")

customer_revenue = (
    df_clean
    .groupby("Customer_Type")["Estimated_Revenue"]
    .agg(["sum", "mean", "median"])
    .sort_values("sum", ascending=False)
)

display(customer_revenue)

# ============================================================
# 14. MONTHLY BOOKING TREND
# ============================================================

monthly_bookings = (
    df_clean
    .groupby(
        df_clean["Arrival_Date"].dt.to_period("M")
    )
    .size()
)

plt.figure(figsize=(14,6))

monthly_bookings.plot()

plt.title("Monthly Hotel Booking Trend")
plt.xlabel("Month")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ============================================================
# 15. MONTHLY CANCELLATION TREND
# ============================================================

monthly_cancel = (
    df_clean
    .groupby(
        df_clean["Arrival_Date"].dt.to_period("M")
    )["Is_Canceled"]
    .mean()
    .mul(100)
)

plt.figure(figsize=(14,6))

monthly_cancel.plot()

plt.title("Monthly Cancellation Rate Trend")
plt.xlabel("Month")
plt.ylabel("Cancellation Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ============================================================
# 16. ARRIVAL MONTH ANALYSIS
# ============================================================

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

monthly_avg = (
    df_clean
    .groupby("Arrival_Month_Name")["ADR"]
    .mean()
    .reindex(month_order)
)

plt.figure(figsize=(12,6))

monthly_avg.plot(kind="bar")

plt.title("Average ADR by Arrival Month")
plt.xlabel("Month")
plt.ylabel("Average ADR")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ============================================================
# 17. ROOM TYPE ANALYSIS
# ============================================================

room_analysis = (
    df_clean
    .groupby("Room_Type_Reserved")
    .agg(
        Bookings=("Booking_ID", "count"),
        Avg_ADR=("ADR", "mean"),
        Avg_Nights=("Total_Nights", "mean"),
        Cancellation_Rate=("Is_Canceled", "mean")
    )
)

room_analysis["Cancellation_Rate"] *= 100

room_analysis = room_analysis.sort_values(
    "Bookings",
    ascending=False
)

print("\n========== ROOM TYPE ANALYSIS ==========")
display(room_analysis)

plt.figure(figsize=(10,5))

sns.barplot(
    data=room_analysis.reset_index(),
    x="Room_Type_Reserved",
    y="Bookings"
)

plt.title("Bookings by Reserved Room Type")
plt.xlabel("Room Type")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()

# ============================================================
# 18. SPECIAL REQUEST ANALYSIS
# ============================================================

plt.figure(figsize=(9,6))

sns.boxplot(
    data=df_clean,
    x="Cancellation_Label",
    y="Total_Special_Requests"
)

plt.title("Special Requests vs Cancellation")
plt.xlabel("Booking Status")
plt.ylabel("Number of Special Requests")
plt.tight_layout()
plt.show()

# ============================================================
# 19. REPEATED GUEST ANALYSIS
# ============================================================

repeat_analysis = (
    df_clean
    .groupby("Is_Repeated_Guest")
    .agg(
        Bookings=("Booking_ID", "count"),
        Avg_ADR=("ADR", "mean"),
        Cancellation_Rate=("Is_Canceled", "mean"),
        Avg_Revenue=("Estimated_Revenue", "mean")
    )
)

repeat_analysis["Cancellation_Rate"] *= 100

print("\n========== REPEATED GUEST ANALYSIS ==========")
display(repeat_analysis)

# ============================================================
# 20. CORRELATION ANALYSIS
# ============================================================

numeric_df = df_clean.select_dtypes(
    include=np.number
)

correlation = numeric_df.corr()

print("\n========== CORRELATION MATRIX ==========")

display(correlation)

plt.figure(figsize=(16,12))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap - Hotel Booking Dataset")
plt.tight_layout()
plt.show()

# ============================================================
# 21. IMPORTANT CORRELATIONS WITH CANCELLATION
# ============================================================

if "Is_Canceled" in correlation.columns:

    cancellation_corr = (
        correlation["Is_Canceled"]
        .sort_values(
            ascending=False
        )
    )

    print("\n========== CORRELATION WITH CANCELLATION ==========")

    display(cancellation_corr)

# ============================================================
# 22. TOP COUNTRIES BY BOOKINGS
# ============================================================

country_bookings = (
    df_clean["Country"]
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10,6))

sns.barplot(
    x=country_bookings.values,
    y=country_bookings.index
)

plt.title("Top 10 Countries by Number of Bookings")
plt.xlabel("Bookings")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

# ============================================================
# 23. TOP REVENUE GENERATING SEGMENTS
# ============================================================

segment_revenue = (
    df_clean
    .groupby("Market_Segment")["Estimated_Revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10,6))

sns.barplot(
    x=segment_revenue.values,
    y=segment_revenue.index
)

plt.title("Estimated Revenue by Market Segment")
plt.xlabel("Estimated Revenue")
plt.ylabel("Market Segment")
plt.tight_layout()
plt.show()

# ============================================================
# 24. KEY BUSINESS METRICS
# ============================================================

total_bookings = len(df_clean)

total_revenue = df_clean["Estimated_Revenue"].sum()

average_adr = df_clean["ADR"].mean()

cancellation_rate = df_clean["Is_Canceled"].mean() * 100

average_stay = df_clean["Total_Nights"].mean()

average_lead_time = df_clean["Lead_Time_Days"].mean()

average_satisfaction = df_clean["Satisfaction_Score"].mean()

print("\n" + "="*60)
print("              EXECUTIVE BUSINESS METRICS")
print("="*60)

print(f"Total Bookings       : {total_bookings:,.0f}")
print(f"Estimated Revenue    : {total_revenue:,.2f}")
print(f"Average ADR          : {average_adr:,.2f}")
print(f"Cancellation Rate    : {cancellation_rate:.2f}%")
print(f"Average Stay         : {average_stay:.2f} nights")
print(f"Average Lead Time    : {average_lead_time:.2f} days")
print(f"Average Satisfaction : {average_satisfaction:.2f}")

# ============================================================
# 25. AUTOMATIC BUSINESS INSIGHTS
# ============================================================

highest_booking_hotel = (
    df_clean["Hotel_Type"]
    .value_counts()
    .idxmax()
)

highest_revenue_hotel = (
    df_clean
    .groupby("Hotel_Type")["Estimated_Revenue"]
    .sum()
    .idxmax()
)

highest_cancel_hotel = (
    df_clean
    .groupby("Hotel_Type")["Is_Canceled"]
    .mean()
    .idxmax()
)

highest_cancel_segment = (
    df_clean
    .groupby("Market_Segment")["Is_Canceled"]
    .mean()
    .idxmax()
)

highest_revenue_segment = (
    df_clean
    .groupby("Market_Segment")["Estimated_Revenue"]
    .sum()
    .idxmax()
)

highest_adr_hotel = (
    df_clean
    .groupby("Hotel_Type")["ADR"]
    .mean()
    .idxmax()
)

print("\n" + "="*60)
print("                  KEY BUSINESS INSIGHTS")
print("="*60)

print(f"""
1. {highest_booking_hotel} has the highest number of bookings,
   indicating stronger demand compared with other hotel types.

2. {highest_revenue_hotel} generates the highest estimated
   revenue among the hotel categories.

3. {highest_cancel_hotel} has the highest cancellation rate,
   indicating a potential revenue leakage and demand-management issue.

4. {highest_cancel_segment} has the highest cancellation rate
   among market segments and should be monitored closely.

5. {highest_revenue_segment} is the highest revenue-generating
   market segment and can be prioritized for targeted campaigns.

6. {highest_adr_hotel} records the highest average ADR,
   suggesting stronger pricing potential in this hotel category.
""")

# ============================================================
# 26. MANAGEMENT RECOMMENDATIONS
# ============================================================

print("\n" + "="*60)
print("              MANAGEMENT RECOMMENDATIONS")
print("="*60)

print("""
1. Implement dynamic pricing:
   Adjust ADR based on season, demand, booking lead time,
   and hotel type to maximize revenue.

2. Reduce cancellation risk:
   Identify high-risk bookings using lead time, market segment,
   deposit type, and previous cancellation behavior.

3. Strengthen high-value market segments:
   Focus marketing budgets on segments generating higher
   revenue while maintaining acceptable cancellation levels.

4. Improve direct bookings:
   Encourage customers to book directly through loyalty
   benefits, discounts, flexible packages, and exclusive offers.

5. Develop customer retention programs:
   Target repeated guests with loyalty rewards and
   personalized offers to increase repeat bookings.

6. Optimize room inventory:
   Use booking trends and room-type demand to allocate
   room inventory efficiently and minimize revenue loss.

7. Monitor seasonal demand:
   Use monthly booking and ADR trends to plan staffing,
   promotions, pricing, and room availability in advance.
""")

# ============================================================
# 27. FINAL DATA QUALITY CHECK
# ============================================================

print("\n========== FINAL DATA QUALITY CHECK ==========")

print("Final dataset shape:", df_clean.shape)

print("\nMissing values:")
display(
    df_clean.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nDuplicate records:", df_clean.duplicated().sum())

print("\nFinal data types:")
display(df_clean.dtypes)

# ============================================================
# 28. SAVE CLEANED DATASET
# ============================================================

output_file = "Cleaned_Hotel_Booking_EDA_Dataset.csv"

df_clean.to_csv(
    output_file,
    index=False
)

print("\nCleaned dataset saved as:", output_file)

# ============================================================
# END OF ANALYSIS
# ============================================================

print("\n" + "="*60)
print("       HOTEL BOOKING EDA COMPLETED SUCCESSFULLY")
print("="*60)